# Policy-Aware RAG Evaluation Analysis

This notebook analyzes the current Function App output, including per-step latency and token estimates captured in the single request-level audit log.

The current app exposes /api/rag and stores all step telemetry under a single request record in AuditStorage, which makes it possible to measure retrieval, policy evaluation, and response guardrail timing individually.


In [13]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv

try:
    from datasets import Dataset
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from ragas import evaluate
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.llms import LangchainLLMWrapper
    from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

    load_dotenv(find_dotenv())
    foundry_endpoint = os.getenv('FOUNDRY_ENDPOINT')
    foundry_api_key = os.getenv('FOUNDRY_API_KEY') 
    foundry_chat_model = os.getenv('FOUNDRY_CHAT_MODEL')
    foundry_embedding_model = os.getenv('FOUNDRY_EMBEDDING_MODEL')

    if not foundry_endpoint or not foundry_api_key:
        raise ValueError('FOUNDRY_ENDPOINT and FOUNDRY_API_KEY must be configured')

    foundry_llm = LangchainLLMWrapper(
        ChatOpenAI(
            model=foundry_chat_model,
            api_key=foundry_api_key,
            base_url=foundry_endpoint,
            temperature=0,
        )
    )
    foundry_embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(
            model=foundry_embedding_model,
            api_key=foundry_api_key,
            base_url=foundry_endpoint,
        )
    )
    ragas_metrics = [
        AnswerRelevancy(llm=foundry_llm, embeddings=foundry_embeddings),
        Faithfulness(llm=foundry_llm),
    ]
    reference_metrics = [
        ContextPrecision(llm=foundry_llm),
        ContextRecall(llm=foundry_llm),
    ]
    HAS_RAGAS = True
except Exception:
    HAS_RAGAS = False

RESULTS_PATH = ("evaluation_results_2026-09-06T15-10-08Z.json")
print(f"Results file: {RESULTS_PATH}")
print(f"Has RAGAS: {HAS_RAGAS}")

C:\Users\dillo\AppData\Local\Temp\ipykernel_13616\2689987193.py:14: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
C:\Users\dillo\AppData\Local\Temp\ipykernel_13616\2689987193.py:14: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
C:\Users\dillo\AppData\Local\Temp\ipykernel_13616\2689987193.py:14: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from 

Results file: evaluation_results_2026-09-06T15-10-08Z.json
Has RAGAS: True


C:\Users\dillo\AppData\Local\Temp\ipykernel_13616\2689987193.py:33: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  foundry_embeddings = LangchainEmbeddingsWrapper(


In [14]:
def load_results(path):
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    return payload.get('results', [])

records = load_results(RESULTS_PATH)
df = pd.DataFrame(records)
display(df[['case_type', 'status_code', 'expected_outcome', 'actual_outcome', 'passed', 'http_latency_ms', 'total_step_latency_ms']].head())


,case_type,status_code,expected_outcome,actual_outcome,passed,http_latency_ms,total_step_latency_ms
0,allow_observer_metadata,200,allow,allow,True,45853,0
1,allow_observer_routing,200,allow,allow,True,40481,0
2,allow_observer_triage,200,allow,allow,True,40364,0
3,allow_support_customer,200,allow,allow,True,41995,0
4,allow_support_case,200,allow,allow,True,42477,0


In [15]:
step_rows = []
for row in df.itertuples():
    for step in getattr(row, 'step_metrics', []):
        step_rows.append({
            'case_type': row.case_type,
            'stepName': step.get('stepName'),
            'executionStatus': step.get('executionStatus'),
            'latency_ms': step.get('latency_ms', 0),
            'query_tokens': step.get('query_tokens', 0),
            'answer_tokens': step.get('answer_tokens', 0),
            'document_count': step.get('document_count'),
        })

steps_df = pd.DataFrame(step_rows)
display(steps_df.groupby('stepName').agg(step_count=('stepName', 'size'), avg_latency_ms=('latency_ms', 'mean')).sort_values('avg_latency_ms', ascending=False))


,step_count,avg_latency_ms
stepName,,
IntentValidation,99,0.0
OutputRedaction,91,0.0


In [16]:
summary = df.groupby('case_type').agg(
    total=('question', 'size'),
    passed=('passed', 'sum'),
    pass_rate=('passed', 'mean'),
    avg_http_latency_ms=('http_latency_ms', 'mean'),
    avg_step_latency_ms=('total_step_latency_ms', 'mean'),
).reset_index()
summary['pass_rate'] = summary['pass_rate'].round(3)
summary['avg_http_latency_ms'] = summary['avg_http_latency_ms'].round(1)
summary['avg_step_latency_ms'] = summary['avg_step_latency_ms'].round(1)
display(summary.sort_values('pass_rate'))


,case_type,total,passed,pass_rate,avg_http_latency_ms,avg_step_latency_ms
10,allow_support_incident,8,6,0.750,21881.6,0.0
3,allow_observer_triage,9,8,0.889,23210.8,0.0
2,allow_observer_routing,9,8,0.889,29038.0,0.0
1,allow_observer_metadata,9,9,1.000,26752.0,0.0
4,allow_privacy_compliance,8,8,1.000,14003.2,0.0
5,allow_privacy_fraud,8,8,1.000,34561.5,0.0
6,allow_privacy_privacy,8,8,1.000,24565.6,0.0
0,allow_admin_full_access,8,8,1.000,6340.4,0.0
7,allow_privacy_security,8,8,1.000,29539.1,0.0
8,allow_support_case,8,8,1.000,17712.0,0.0


In [18]:
if HAS_RAGAS:
    ragas_rows = []
    for row in df.itertuples():
        record = row._asdict()
        answer = str(record.get('response', {}).get('answer', '') if isinstance(record.get('response'), dict) else '')
        sources = record.get('response', {}).get('sources', []) if isinstance(record.get('response'), dict) else []
        reference = next(
            (record.get(field) for field in ('reference', 'ground_truth', 'reference_answer') if record.get(field)),
            None,
        )
        if not answer:
            continue

        dataset_values = {
            'question': [record.get('question', '')],
            'answer': [answer],
            'contexts': [[str(item) for item in sources]],
        }
        metrics = ragas_metrics
        if reference:
            dataset_values['reference'] = [str(reference)]
            metrics = ragas_metrics + reference_metrics
        dataset = Dataset.from_dict(dataset_values)
        scores = evaluate(dataset, metrics=metrics)
        score_values = scores.scores[0]
        ragas_rows.append({
            'case_type': record.get('case_type'),
            **{k: float(v) for k, v in score_values.items() if isinstance(v, (int, float))},
        })
    ragas_df = pd.DataFrame(ragas_rows)
    display(ragas_df.groupby('case_type').mean(numeric_only=True).round(3))
else:
    print('RAGAS is not installed or Azure Foundry settings are unavailable; showing proxy pass-rate summary instead.')
    display(df.groupby('case_type').agg(pass_rate=('passed', 'mean')).round(3))

Evaluating: 100%|██████████| 2/2 [00:13<00:00,  6.61s/it]


,answer_relevancy,faithfulness
case_type,,
allow_admin_full_access,0.473,0.298
allow_observer_metadata,0.740,0.472
allow_observer_routing,0.655,0.093
allow_observer_triage,0.621,0.284
allow_privacy_compliance,0.830,0.379
allow_privacy_fraud,0.599,0.477
allow_privacy_privacy,0.820,0.515
allow_privacy_security,0.438,0.172
allow_support_case,0.578,0.434
